# 02 — Feature Engineering

Converts each drug pair's SMILES strings into concatenated Morgan fingerprints. This is the slow step (RDKit over ~192k rows) — expect several minutes.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

df = pd.read_pickle('../data/df_grouped.pkl')
print(df.shape)


In [ ]:
from src.features import smiles_to_fp

fp1 = np.vstack(df['Drug1'].apply(smiles_to_fp))
fp2 = np.vstack(df['Drug2'].apply(smiles_to_fp))
X = np.concatenate([fp1, fp2], axis=1)
y = df['Y_grouped'].values

print(X.dtype, X.nbytes / 1e9, "GB")


## Check for invalid SMILES

Malformed SMILES silently become zero vectors rather than raising — worth knowing how many, since a large number would need a different handling strategy than a simple drop.

In [ ]:
n_invalid_drug1 = (fp1.sum(axis=1) == 0).sum()
n_invalid_drug2 = (fp2.sum(axis=1) == 0).sum()
print(f"Invalid SMILES — Drug1: {n_invalid_drug1} ({n_invalid_drug1/len(df):.2%})")
print(f"Invalid SMILES — Drug2: {n_invalid_drug2} ({n_invalid_drug2/len(df):.2%})")


## Save fingerprints for the training notebook

In [ ]:
np.save('../data/X_fingerprints.npy', X)
np.save('../data/y_labels.npy', y)
print("Saved.")
